In [4]:
"""
NASCAR Cup Series Comprehensive Scraper
========================================
Source  : racing-reference.info
Seasons : 2025 & 2026
Output  : nascar_2025_2026.csv

HOW TO RUN:
  1. Install dependencies:
       pip install requests beautifulsoup4

  2. Run this script:
       python nascar_scraper.py

  The CSV file will appear in the same folder as this script.
"""

import requests
from bs4 import BeautifulSoup
import csv
import time
import re
import os

# ──────────────────────────────────────────────────────────────────
# CONFIGURATION  (edit these if needed)
# ──────────────────────────────────────────────────────────────────
YEARS      = [2025, 2026]  # seasons to scrape
SERIES     = "W"           # W = NASCAR Cup Series
MAX_RACES  = 40            # upper limit; we stop early when no more races exist
OUTPUT     = "data/raw/nascar_2025_2026.csv"
DELAY      = 1.5           # seconds to wait between requests 

BASE = "https://www.racing-reference.info"
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    )
}

# ──────────────────────────────────────────────────────────────────
# HELPERS
# ──────────────────────────────────────────────────────────────────
def get_soup(url: str) -> BeautifulSoup | None:
    """Fetch a URL and return a BeautifulSoup object, or None on failure."""
    try:
        resp = requests.get(url, headers=HEADERS, timeout=15)
        if resp.status_code != 200:
            return None
        return BeautifulSoup(resp.text, "html.parser")
    except Exception as e:
        print(f"    ⚠  Request failed ({url}): {e}")
        return None


def clean(text: str) -> str:
    """Strip extra whitespace from a string."""
    return " ".join(text.split()) if text else ""


# ──────────────────────────────────────────────────────────────────
# PARSE RACE METADATA  (from the top of each race results page)
# ──────────────────────────────────────────────────────────────────
def parse_race_meta(soup: BeautifulSoup) -> dict:
    """
    Extract race-level information:
      race_name, race_date, track, track_type, track_miles,
      total_laps, caution_flags, caution_laps, lead_changes,
      avg_speed_mph, pole_speed_mph, margin_of_victory
    """
    meta = {}
    content = soup.find(id="content") or soup.body
    if content is None:
        return meta

    full_text = content.get_text(" ", strip=True)

    # Race name — look for the <h2> or <li> that contains the race title
    h1 = soup.find("h1")
    if h1:
        meta["race_name"] = clean(h1.get_text())
    else:
        # fallback: grab it from the page title
        title_tag = soup.find("title")
        meta["race_name"] = clean(title_tag.get_text()).split("|")[0].strip() if title_tag else ""

    # Date  e.g. "Sunday, June 21, 2026"
    date_m = re.search(
        r"(Monday|Tuesday|Wednesday|Thursday|Friday|Saturday|Sunday),\s+[\w]+ \d+,\s+\d{4}",
        full_text
    )
    meta["race_date"] = date_m.group(0) if date_m else ""

    # Track name (linked)
    track_tag = content.find("a", href=re.compile(r"/tracks/"))
    if track_tag:
        meta["track"] = clean(track_tag.get_text())

    # "75 laps on a 3.400 mile street course (255.000 miles)"
    laps_m = re.search(
        r"(\d+)\s+laps\s+on\s+a\s+([\d.]+)\s+mile\s+([\w\s]+?)\s*\(",
        full_text
    )
    if laps_m:
        meta["total_laps"]  = laps_m.group(1)
        meta["track_miles"] = laps_m.group(2)
        meta["track_type"]  = laps_m.group(3).strip()

    # Cautions
    c_m = re.search(r"Cautions:\s+(\d+)\s+for\s+(\d+)\s+laps", full_text)
    if c_m:
        meta["caution_flags"] = c_m.group(1)
        meta["caution_laps"]  = c_m.group(2)

    # Lead changes
    lc_m = re.search(r"Lead changes:\s+(\d+)", full_text)
    meta["lead_changes"] = lc_m.group(1) if lc_m else ""

    # Average speed
    spd_m = re.search(r"Average speed:\s+([\d.]+)\s+mph", full_text)
    meta["avg_speed_mph"] = spd_m.group(1) if spd_m else ""

    # Pole speed
    pole_m = re.search(r"Pole speed:\s+([\d.]+)\s+mph", full_text)
    meta["pole_speed_mph"] = pole_m.group(1) if pole_m else ""

    # Margin of victory
    mov_m = re.search(
    r"Margin of victory:\s+(.*?)(?=\s+(?:Attendance|Lead|Average|Cautions|Pole))",
    full_text
)
    meta["margin_of_victory"] = clean(mov_m.group(1)) if mov_m else ""

    # Attendance
    att_m = re.search(r"Attendance:\s+([\d,n/a]+)", full_text, re.IGNORECASE)
    meta["attendance"] = clean(att_m.group(1)) if att_m else ""

    return meta


# ──────────────────────────────────────────────────────────────────
# PARSE RESULTS TABLE  (per-driver finishing data)
# ──────────────────────────────────────────────────────────────────
def parse_results_table(soup: BeautifulSoup) -> list[dict]:
    rows = []
    tables = soup.find_all("table")

    target = None
    # Race results table has columns points tables don't: "St", "#", "Sponsor / Owner", "Car", "Laps"
    RACE_REQUIRED_COLS = {"Pos", "St", "#", "Driver", "Led", "Car"}

    for tbl in tables:
        th_texts = [clean(th.get_text()) for th in tbl.find("thead").find_all("th")] \
                if tbl.find("thead") else \
                [clean(th.get_text()) for th in (tbl.find("tr") or {}).find_all("th")]
        if RACE_REQUIRED_COLS.issubset(set(th_texts)):
            target = tbl
            headers = th_texts
            break

    if target is None:
        return rows

    # build a lookup: column name → index
    col = {name: i for i, name in enumerate(headers)}
    # col now looks like: {"": 0, "Pos": 1, "St": 2, "#": 3, "Driver": 4, ...}

    for tr in target.find_all("tr"):
        cells = tr.find_all("td")
        min_cells = max(col.values()) + 1
        if len(cells) < min_cells:
            continue

        pos = clean(cells[col["Pos"]].get_text())
        if not pos.isdigit():
            continue
        
        if not clean(cells[col["St"]].get_text()).isdigit():
            continue

        driver_a = cells[col["Driver"]].find("a")
        driver   = clean(driver_a.get_text()) if driver_a else clean(cells[col["Driver"]].get_text())

        sponsor_cell = cells[col["Sponsor / Owner"]]
        b_tag   = sponsor_cell.find("b")
        a_tag   = sponsor_cell.find("a")
        
        # Use <b> tag text if available, otherwise take the whole cell
        sponsor_text = clean(b_tag.get_text()) if b_tag else clean(sponsor_cell.get_text())

        rows.append({
            "finish_pos"     : pos,
            "start_pos"      : clean(cells[col["St"]].get_text()),
            "car_number"     : clean(cells[col["#"]].get_text()),
            "driver"         : driver,
            "sponsor"        : sponsor_text,
            "owner"          : clean(a_tag.get_text()) if a_tag else "",
            "car_make"       : clean(cells[col["Car"]].get_text()),
            "laps_completed" : clean(cells[col["Laps"]].get_text()),
            "status"         : clean(cells[col["Status"]].get_text()),
            "laps_led"       : clean(cells[col["Led"]].get_text()),
            "points"         : clean(cells[col["Pts"]].get_text()),
        })

    return rows


# ──────────────────────────────────────────────────────────────────
# PARSE LOOP DATA  (advanced per-driver stats from NASCAR's loop data)
# ──────────────────────────────────────────────────────────────────
def parse_loop_data(soup: BeautifulSoup) -> dict[str, dict]:
    """
    Parse the Loop Data page.
    Returns a dict: { car_number_str -> {column: value, ...} }
    """
    result = {}
    if soup is None:
        return result

    tables = soup.find_all("table")
    for tbl in tables:
        th_texts = [clean(th.get_text()) for th in tbl.find_all("th")]
        # Loop data tables contain columns like Pass, Speed, Rating, etc.
        if not any(kw in " ".join(th_texts) for kw in ("Pass", "Speed", "Rating", "Green", "Fast")):
            continue

        for tr in tbl.find_all("tr"):
            cells = tr.find_all("td")
            if len(cells) < 4:
                continue
            row_data = {}
            for i, th in enumerate(th_texts):
                if i < len(cells):
                    row_data[th] = clean(cells[i].get_text())
            # Key by car number
            car_key = row_data.get("#") or row_data.get("Car") or ""
            if car_key:
                result[car_key] = row_data

    return result


# ──────────────────────────────────────────────────────────────────
# CSV COLUMN ORDER
# ──────────────────────────────────────────────────────────────────
CSV_COLUMNS = [
    # ── Race-level ──────────────────
    "year",
    "race_number",
    "race_name",
    "race_date",
    "track",
    "track_type",
    "track_miles",
    "total_laps",
    "caution_flags",
    "caution_laps",
    "lead_changes",
    "avg_speed_mph",
    "pole_speed_mph",
    "margin_of_victory",
    "attendance",
    # ── Driver-level ─────────────────
    "finish_pos",
    "start_pos",
    "car_number",
    "driver",
    "sponsor",
    "owner",
    "car_make",
    "laps_completed",
    "status",
    "laps_led",
    "points",
    # ── Loop Data (if available) ─────
    "loop_avg_speed",
    "loop_green_flag_passes",
    "loop_quality_passes",
    "loop_fastest_lap",
    "loop_driver_rating",
]


# ──────────────────────────────────────────────────────────────────
# MAIN SCRAPING LOOP
# ──────────────────────────────────────────────────────────────────
def main():
    all_rows = []

    for year in YEARS:
        print(f"\n{'='*55}")
        print(f"  Scraping {year} NASCAR Cup Series  (series={SERIES})")
        print(f"{'='*55}")
        #breakpoint()  Debugging breakpoint; remove or comment out in production
        for race_num in range(1, MAX_RACES + 1):
            race_id      = f"{year}-{race_num:02d}"
            results_url  = f"{BASE}/race-results/{race_id}/{SERIES}"
            loop_url     = f"{BASE}/loopdata/{race_id}/{SERIES}"

            print(f"  [{race_id}]  Fetching results …")
            soup_results = get_soup(results_url)

            # Stop when we run out of races (returns None or a 404-like page)
            if soup_results is None:
                print("  → No response; stopping for this year.")
                break

            page_text = soup_results.get_text()
            if "Page not found" in page_text or len(page_text) < 500:
                print("  → Empty / 404 page; stopping for this year.")
                break

            meta    = parse_race_meta(soup_results)
            dr_rows = parse_results_table(soup_results)

            if not dr_rows:
                print(f"  → No result rows found (race may not have happened yet). Stopping.")
                continue

            # Fetch loop data (extra stats) for this race
            time.sleep(DELAY)
            print(f"  [{race_id}]  Fetching loop data …")
            soup_loop    = get_soup(loop_url)
            loop_by_car  = parse_loop_data(soup_loop)

            # Merge race meta + driver rows + loop data → one flat row per driver
            for dr in dr_rows:
                loop = loop_by_car.get(dr["car_number"], {})

                flat = {
                    "year"        : year,
                    "race_number" : race_num,
                    **meta,
                    **dr,
                    # try several possible column names from loop data
                    "loop_avg_speed"         : loop.get("Avg Sp") or loop.get("Avg Speed") or "",
                    "loop_green_flag_passes" : loop.get("GF Pass") or loop.get("Green Flag Passes") or "",
                    "loop_quality_passes"    : loop.get("Qual Pass") or loop.get("Quality Passes") or "",
                    "loop_fastest_lap"       : loop.get("Fast Lap") or loop.get("Fastest Lap") or "",
                    "loop_driver_rating"     : loop.get("Rating") or loop.get("Driver Rating") or "",
                }
                all_rows.append({col: flat.get(col, "") for col in CSV_COLUMNS})

            print(f"  ✓  {len(dr_rows)} drivers | Race: {meta.get('race_name', '?')}")
            time.sleep(DELAY)

    # ── Write CSV ───────────────────────────────────────────────
    with open(OUTPUT, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=CSV_COLUMNS)
        writer.writeheader()
        writer.writerows(all_rows)

    print(f"\n{'='*55}")
    print(f"  ✅  DONE!  {len(all_rows)} total rows written to:")
    print(f"      {os.path.abspath(OUTPUT)}")
    print(f"{'='*55}")


if __name__ == "__main__":
    main()


  Scraping 2025 NASCAR Cup Series  (series=W)
  [2025-01]  Fetching results …
  [2025-01]  Fetching loop data …
  ✓  41 drivers | Race: 2025 Daytona 500
  [2025-02]  Fetching results …
  [2025-02]  Fetching loop data …
  ✓  39 drivers | Race: 2025 Ambetter Health 400
  [2025-03]  Fetching results …
  [2025-03]  Fetching loop data …
  ✓  37 drivers | Race: 2025 EchoPark Automotive Grand Prix
  [2025-04]  Fetching results …
  [2025-04]  Fetching loop data …
  ✓  37 drivers | Race: 2025 Shriners Childrens 500
  [2025-05]  Fetching results …
  [2025-05]  Fetching loop data …
  ✓  36 drivers | Race: 2025 Pennzoil 400
  [2025-06]  Fetching results …
  [2025-06]  Fetching loop data …
  ✓  37 drivers | Race: 2025 Straight Talk Wireless 400
  [2025-07]  Fetching results …
  [2025-07]  Fetching loop data …
  ✓  38 drivers | Race: 2025 Cook Out 400
  [2025-08]  Fetching results …
  [2025-08]  Fetching loop data …
  ✓  38 drivers | Race: 2025 Goodyear 400
  [2025-09]  Fetching results …
  [2025-0

In [3]:
import requests
r = requests.get("https://www.racing-reference.info/race-results/2025-01/W", 
                 headers=HEADERS, timeout=15)
print(r.status_code, len(r.text))

403 5780
